# Low Pass & High Pass Filters â€” From Scratch
**No built-in filter functions. Non-ideal (realistic) filters implemented manually.**

## 1. Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## 2. Create a Test Signal

We mix a **low-frequency** sine (5 Hz) + a **high-frequency** sine (50 Hz).  
- After Low Pass  â†’ only the 5 Hz part should survive  
- After High Pass â†’ only the 50 Hz part should survive

In [ ]:
fs = 500          # sampling frequency (Hz)
T  = 1.0          # signal duration (seconds)
t  = np.arange(0, T, 1/fs)   # time array

low_freq  = 5     # Hz  â€” the "useful" slow signal
high_freq = 50    # Hz  â€” the "noise" fast signal

signal = np.sin(2 * np.pi * low_freq  * t) + \
         np.sin(2 * np.pi * high_freq * t)

plt.figure(figsize=(12, 3))
plt.plot(t, signal)
plt.title("Original Signal (5 Hz + 50 Hz)")
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.tight_layout()
plt.show()

## 3. Low Pass Filter (from scratch)

**Idea:** In the frequency domain, keep only frequencies **below** the cutoff; zero out the rest.

Steps:
1. Compute the FFT of the signal manually using `np.fft.fft` (this is the math, not a filter library)
2. Build a mask: 1 where |frequency| â‰¤ cutoff, else 0
3. Multiply FFT Ã— mask  â†’ removes high frequencies
4. Inverse FFT back to time domain

> **Non-ideal** because it uses a sharp rectangular mask (abrupt cutoff), which causes some ringing â€” this is realistic and expected.

In [ ]:
def low_pass_filter(signal, fs, cutoff_hz):
    """
    Low pass filter using FFT â€” from scratch, non-ideal.
    Keeps frequencies <= cutoff_hz, removes the rest.
    """
    N = len(signal)

    # Step 1: Compute FFT
    fft_vals = np.fft.fft(signal)

    # Step 2: Get the frequency value for each FFT bin
    freqs = np.fft.fftfreq(N, d=1/fs)

    # Step 3: Build mask â€” keep bins where |freq| <= cutoff
    mask = np.abs(freqs) <= cutoff_hz

    # Step 4: Apply mask (zero out high frequencies)
    fft_filtered = fft_vals * mask

    # Step 5: Inverse FFT to get filtered signal back
    filtered_signal = np.fft.ifft(fft_filtered).real

    return filtered_signal


cutoff = 20   # Hz â€” anything above 20 Hz will be removed

lp_output = low_pass_filter(signal, fs, cutoff)

plt.figure(figsize=(12, 3))
plt.plot(t, lp_output, color='blue')
plt.title(f"Low Pass Filter Output (cutoff = {cutoff} Hz) â€” 5 Hz survives")
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.tight_layout()
plt.show()

## 4. High Pass Filter (from scratch)

**Idea:** Exact opposite of low pass â€” keep frequencies **above** the cutoff; zero out the rest.

Steps are the same, only the mask is flipped: 1 where |frequency| â‰¥ cutoff, else 0.

In [ ]:
def high_pass_filter(signal, fs, cutoff_hz):
    """
    High pass filter using FFT â€” from scratch, non-ideal.
    Keeps frequencies >= cutoff_hz, removes the rest.
    """
    N = len(signal)

    # Step 1: Compute FFT
    fft_vals = np.fft.fft(signal)

    # Step 2: Get frequency for each bin
    freqs = np.fft.fftfreq(N, d=1/fs)

    # Step 3: Build mask â€” keep bins where |freq| >= cutoff
    mask = np.abs(freqs) >= cutoff_hz

    # Step 4: Apply mask (zero out low frequencies)
    fft_filtered = fft_vals * mask

    # Step 5: Inverse FFT
    filtered_signal = np.fft.ifft(fft_filtered).real

    return filtered_signal


cutoff = 20   # Hz â€” anything below 20 Hz will be removed

hp_output = high_pass_filter(signal, fs, cutoff)

plt.figure(figsize=(12, 3))
plt.plot(t, hp_output, color='red')
plt.title(f"High Pass Filter Output (cutoff = {cutoff} Hz) â€” 50 Hz survives")
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.tight_layout()
plt.show()

## 5. Full Comparison Plot

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)

axes[0].plot(t, signal, color='black')
axes[0].set_title("Original Signal (5 Hz + 50 Hz)")
axes[0].set_ylabel("Amplitude")

axes[1].plot(t, lp_output, color='blue')
axes[1].set_title("Low Pass Filter Output â€” 5 Hz only")
axes[1].set_ylabel("Amplitude")

axes[2].plot(t, hp_output, color='red')
axes[2].set_title("High Pass Filter Output â€” 50 Hz only")
axes[2].set_ylabel("Amplitude")
axes[2].set_xlabel("Time (s)")

plt.tight_layout()
plt.show()

## 6. Visualize the Filter Mask in Frequency Domain

This shows **what frequencies are kept vs removed** for each filter.

In [ ]:
N     = len(signal)
freqs = np.fft.fftfreq(N, d=1/fs)
cutoff = 20

# Magnitude of original FFT
fft_mag = np.abs(np.fft.fft(signal)) / N

# Only show positive frequencies for clarity
pos = freqs >= 0

lp_mask = (np.abs(freqs) <= cutoff).astype(float)
hp_mask = (np.abs(freqs) >= cutoff).astype(float)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(freqs[pos], fft_mag[pos], 'k', label='Signal spectrum')
axes[0].fill_between(freqs[pos], lp_mask[pos], alpha=0.3, color='blue', label='LP Mask (keep)')
axes[0].axvline(cutoff, color='blue', linestyle='--', label=f'Cutoff = {cutoff} Hz')
axes[0].set_title("Low Pass Filter Mask")
axes[0].set_xlabel("Frequency (Hz)")
axes[0].set_ylabel("Magnitude")
axes[0].legend()

axes[1].plot(freqs[pos], fft_mag[pos], 'k', label='Signal spectrum')
axes[1].fill_between(freqs[pos], hp_mask[pos], alpha=0.3, color='red', label='HP Mask (keep)')
axes[1].axvline(cutoff, color='red', linestyle='--', label=f'Cutoff = {cutoff} Hz')
axes[1].set_title("High Pass Filter Mask")
axes[1].set_xlabel("Frequency (Hz)")
axes[1].legend()

plt.tight_layout()
plt.show()

## Summary

| | Low Pass | High Pass |
|---|---|---|
| **Keeps** | Low frequencies (slow signals) | High frequencies (fast signals) |
| **Removes** | High frequencies | Low frequencies |
| **Mask condition** | `abs(freq) <= cutoff` | `abs(freq) >= cutoff` |
| **Use case** | Noise removal (noise is usually high freq) | Edge detection, isolating fast changes |

**Why non-ideal?** A perfect (ideal) filter would have a perfectly vertical cutoff edge.  
Our rectangular mask is non-ideal because in practice it causes **Gibbs phenomenon** (slight ringing near the cutoff).  
Real-world filters always have a transition band â€” they are non-ideal by nature.